# Mecab 사용자 사전용 csv 파일 구성

In [1]:
import pandas as pd
import jamo # 종성 유무 판단을 위해 필요할 수 있음 (pip install jamo)

def has_coda(word):
    """
    한글 단어의 마지막 글자에 받침(종성)이 있는지 확인하는 함수
    Mecab 사전 포맷에서 종성 유무(T/F)가 필요함
    """
    last_char = word[-1]
    # 한글 유니코드 범위: 0xAC00 ~ 0xD7A3
    if 0xAC00 <= ord(last_char) <= 0xD7A3:
        # (유니코드 - 0xAC00) % 28 이 0이면 받침 없음
        return (ord(last_char) - 0xAC00) % 28 != 0
    return False

def create_mecab_user_dic_csv(term_list, output_path="user_dic.csv"):
    """
    법률 용어 리스트를 Mecab 사용자 사전 포맷(CSV)으로 저장
    """
    user_dic_data = []
    
    for term in term_list:
        # 전처리: 공백 제거, 특수문자 제거 등 필요 시 추가
        clean_term = term.strip()
        if not clean_term: continue
        
        # 종성 유무 판단
        coda_flag = "T" if has_coda(clean_term) else "F"
        
        # Mecab 사전 포맷 (표층형, 좌문맥ID, 우문맥ID, 비용, 품사, 의미분류, 종성유무, 읽기, 타입, 첫품사, 끝품사, 표현)
        # NNP(고유명사)로 강제 지정, 비용(Cost)은 0으로 낮춰서 우선순위 높임
        line = [clean_term, 0, 0, 0, "NNP", "*", coda_flag, clean_term, "*", "*", "*", "*"]
        user_dic_data.append(line)
    
    # CSV 저장 (헤더 없음)
    df = pd.DataFrame(user_dic_data)
    df.to_csv(output_path, index=False, header=False, encoding='utf-8')
    print(f"Mecab 사용자 사전 파일 생성 완료: {output_path} ({len(user_dic_data)}개 용어)")


In [2]:
df = pd.read_csv("data\법령용어목록_전체.csv")

In [3]:
create_mecab_user_dic_csv(df)

Mecab 사용자 사전 파일 생성 완료: user_dic.csv (5개 용어)
